<a href="https://colab.research.google.com/github/chaeminju/26-1IAP/blob/main/26_1IAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 필수 라이브러리 설치
!pip install -q tensorflow tensorflow-hub opencv-python ultralytics
!wget -nc -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -n -q annotations_trainval2017.zip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.5 MB/s eta 0:00:00


In [ ]:
import json
import os
import random
import time
import numpy as np
import cv2
import tensorflow as tf
import tensorflow_hub as hub
from ultralytics import YOLO

# 재현성을 위한 시드 고정
random.seed(42)
ANNOT_PATH = "annotations/person_keypoints_val2017.json"
SAVE_DIR = "images"
os.makedirs(SAVE_DIR, exist_ok=True)

with open(ANNOT_PATH, 'r') as f:
    data = json.load(f)

# 사람(keypoints)이 있는 이미지 100장 샘플링
valid_image_ids = set([ann['image_id'] for ann in data['annotations'] if ann['num_keypoints'] > 0])
filtered_images = [img for img in data['images'] if img['id'] in valid_image_ids]
sampled_images = random.sample(filtered_images, 100)

print("테스트용 이미지 100장 준비 중...")
for img_info in sampled_images:
    file_name = img_info['file_name']
    img_path = os.path.join(SAVE_DIR, file_name)
    if not os.path.exists(img_path):
        import requests
        img_data = requests.get(f"http://images.cocodataset.org/val2017/{file_name}").content
        with open(img_path, 'wb') as f:
            f.write(img_data)
print("다운로드 완료!\n")

# ==========================================
# 1. 평가 지표 함수 정의 (정확도 및 속도)
# ==========================================

# OKS (Object Keypoint Similarity) 가중치 (COCO 표준)
COCO_SIGMAS = np.array([0.026, 0.025, 0.025, 0.035, 0.035, 0.079, 0.079, 0.072, 0.072, 0.062, 0.062, 0.107, 0.107, 0.087, 0.087, 0.089, 0.089])

def compute_oks(pred_kp, gt_kp, bbox_area):
    """관절 부위별 난이도를 반영한 정확도 점수 (OKS) 계산"""
    if pred_kp is None or len(pred_kp) == 0: return 0.0
    oks_sum, valid_count = 0, 0

    for i in range(17):
        gt_x, gt_y, vis = gt_kp[i]
        if vis > 0:
            pred_x, pred_y = pred_kp[i][0], pred_kp[i][1]
            dist_sq = (pred_x - gt_x)**2 + (pred_y - gt_y)**2
            kappa = 2 * (COCO_SIGMAS[i] ** 2)
            oks_sum += np.exp(-dist_sq / (2 * bbox_area * kappa))
            valid_count += 1
    return oks_sum / valid_count if valid_count > 0 else 0.0

def compute_detailed_pck(pred_kp, gt_kp, bbox, threshold=0.1):
    """보이는 관절과 가려진 관절을 구분하여 정확도(PCK) 계산"""
    if pred_kp is None or len(pred_kp) == 0: return 0.0, 0.0
    norm = max(bbox[2], bbox[3])

    c_vis, t_vis = 0, 0
    c_occ, t_occ = 0, 0

    for i in range(17):
        gt_x, gt_y, vis = gt_kp[i]
        if vis > 0:
            dist = np.linalg.norm([pred_kp[i][0] - gt_x, pred_kp[i][1] - gt_y])
            is_correct = dist < (threshold * norm)

            if vis == 2: # 완전히 보이는 관절
                t_vis += 1
                if is_correct: c_vis += 1
            elif vis == 1: # 옷이나 사물에 가려진 관절
                t_occ += 1
                if is_correct: c_occ += 1

    pck_vis = (c_vis / t_vis) if t_vis > 0 else 1.0
    pck_occ = (c_occ / t_occ) if t_occ > 0 else 1.0
    return pck_vis, pck_occ

def measure_fps(func, *args):
    """함수 실행 시간을 측정하여 FPS로 변환"""
    start = time.time()
    func(*args)
    end = time.time()
    elapsed = end - start
    return 1 / elapsed if elapsed > 0 else 0.0

# ==========================================
# 2. 모델 로드 및 추론 함수
# ==========================================
print("MoveNet 로드 중...")
movenet_model = hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4")
movenet = movenet_model.signatures['serving_default']

def run_movenet(image_path, img_width, img_height):
    img = cv2.imread(image_path)
    img_resized = cv2.resize(img, (192, 192)).astype(np.int32)
    outputs = movenet(tf.expand_dims(img_resized, axis=0))

    keypoints = outputs['output_0'].numpy()[0, 0]
    abs_kp = [[x * img_width, y * img_height, score] for y, x, score in keypoints]
    return np.array(abs_kp)

print("YOLOv8-Pose 로드 중...")
yolo_model = YOLO('yolov8n-pose.pt')

def run_yolo_pose(image_path):
    results = yolo_model(image_path, verbose=False)
    if len(results[0].keypoints) > 0:
        return results[0].keypoints.data[0].cpu().numpy()
    return None

# ==========================================
# 3. 종합 평가 루프 (100장)
# ==========================================
print("\n========== 속도 및 정확도 벤치마크 시작 (100장) ==========")

metrics = {
    'movenet': {'fps': [], 'oks': [], 'pck_vis': [], 'pck_occ': [], 'misses': 0},
    'yolo':    {'fps': [], 'oks': [], 'pck_vis': [], 'pck_occ': [], 'misses': 0}
}

for i, img_info in enumerate(sampled_images):
    path = os.path.join(SAVE_DIR, img_info['file_name'])
    w, h = img_info['width'], img_info['height']

    # 정답(GT) 정보 추출
    gt_anns = [ann for ann in data['annotations'] if ann['image_id'] == img_info['id'] and ann['num_keypoints'] > 0]
    if not gt_anns: continue
    best_ann = max(gt_anns, key=lambda x: x['bbox'][2] * x['bbox'][3])
    gt_kp = np.array(best_ann['keypoints']).reshape(-1, 3)
    bbox = best_ann['bbox']
    area = best_ann['area']

    # 워밍업 (첫 실행 지연 방지)
    if i == 0:
        run_movenet(path, w, h)
        run_yolo_pose(path)

    # 1. MoveNet 평가
    m_pred = run_movenet(path, w, h)
    metrics['movenet']['fps'].append(measure_fps(run_movenet, path, w, h))
    if m_pred is None:
        metrics['movenet']['misses'] += 1
    else:
        metrics['movenet']['oks'].append(compute_oks(m_pred, gt_kp, area))
        p_vis, p_occ = compute_detailed_pck(m_pred, gt_kp, bbox)
        metrics['movenet']['pck_vis'].append(p_vis)
        metrics['movenet']['pck_occ'].append(p_occ)

    # 2. YOLOv8 평가
    y_pred = run_yolo_pose(path)
    metrics['yolo']['fps'].append(measure_fps(run_yolo_pose, path))
    if y_pred is None:
        metrics['yolo']['misses'] += 1
    else:
        metrics['yolo']['oks'].append(compute_oks(y_pred, gt_kp, area))
        p_vis, p_occ = compute_detailed_pck(y_pred, gt_kp, bbox)
        metrics['yolo']['pck_vis'].append(p_vis)
        metrics['yolo']['pck_occ'].append(p_occ)

    # 진행률 출력 (20장 단위)
    if (i + 1) % 20 == 0:
        print(f" ... [{i + 1}/100] 평가 완료")

# ==========================================
# 4. 최종 결과 출력 함수
# ==========================================
def print_report(name, data_dict):
    avg_fps = np.mean(data_dict['fps'])
    miss_rate = (data_dict['misses'] / 100) * 100
    avg_oks = np.mean(data_dict['oks']) * 100
    pck_v = np.mean(data_dict['pck_vis']) * 100
    pck_o = np.mean(data_dict['pck_occ']) * 100

    print(f"\n[{name}]")
    print(f" ⚡ 속도 관점")
    print(f"    - 평균 처리 속도 : {avg_fps:.1f} FPS")
    print(f"    - 객체 탐지 실패율: {miss_rate:.1f} %")
    print(f" 🎯 정확도 관점 (높을수록 좋음)")
    print(f"    - OKS 종합 점수  : {avg_oks:.1f} 점")
    print(f"    - PCK (보이는 관절): {pck_v:.1f} %")
    print(f"    - PCK (가려진 관절): {pck_o:.1f} %")

print("\n" + "="*50)
print(" 🏆 속도 및 정확도 벤치마크 최종 리포트 🏆")
print("="*50)
print_report("MoveNet (Lightning)", metrics['movenet'])
print("-" * 50)
print_report("YOLOv8-Pose (Nano)", metrics['yolo'])
print("="*50)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
테스트용 이미지 100장 준비 중...
다운로드 완료!

MoveNet 로드 중...
YOLOv8-Pose 로드 중...

========== 속도 및 정확도 벤치마크 시작 (100장) ==========
 ... [20/100] 평가 완료
 ... [40/100] 평가 완료
 ... [60/100] 평가 완료
 ... [80/100] 평가 완료
 ... [100/100] 평가 완료

 🏆 속도 및 정확도 벤치마크 최종 리포트 🏆

[MoveNet (Lightning)]
 ⚡ 속도 관점
    - 평균 처리 속도 : 40.1 FPS
    - 객체 탐지 실패율: 0.0 %
 🎯 정확도 관점 (높을수록 좋음)
    - OKS 종합 점수  : 34.2 점
    - PCK (보이는 관절): 50.9 %
    - PCK (가려진 관절): 71.4 %
--------------------------------------------------

[YOLOv8-Pose (Nano)]
 ⚡ 속도 관점
    - 평균 처리 속도 : 31.8 FPS
    - 객체 탐지 실패율: 1.0 %
 🎯 정확도 관점 (높을수록 좋음)
    - OKS 종합 점수  : 50.8 점
    - PCK (보이는 관절): 62.1 %
    - PCK (가려진 관절): 71.5 %


In [ ]:
import json
import os
import random
import time
import numpy as np
import cv2
import tensorflow as tf
import tensorflow_hub as hub
from ultralytics import YOLO

# 재현성을 위한 시드 고정
random.seed(42)
ANNOT_PATH = "annotations/person_keypoints_val2017.json"
SAVE_DIR = "images"
os.makedirs(SAVE_DIR, exist_ok=True)

with open(ANNOT_PATH, 'r') as f:
    data = json.load(f)

# 사람(keypoints)이 있는 이미지 100장 샘플링
valid_image_ids = set([ann['image_id'] for ann in data['annotations'] if ann['num_keypoints'] > 0])
filtered_images = [img for img in data['images'] if img['id'] in valid_image_ids]
sampled_images = random.sample(filtered_images, 100)

print("테스트용 이미지 100장 준비 중...")
for img_info in sampled_images:
    file_name = img_info['file_name']
    img_path = os.path.join(SAVE_DIR, file_name)
    if not os.path.exists(img_path):
        import requests
        img_data = requests.get(f"http://images.cocodataset.org/val2017/{file_name}").content
        with open(img_path, 'wb') as f:
            f.write(img_data)
print("다운로드 완료!\n")

# ==========================================
# 1. 평가 지표 함수 정의 (정확도 및 속도)
# ==========================================

# OKS (Object Keypoint Similarity) 가중치 (COCO 표준)
COCO_SIGMAS = np.array([0.026, 0.025, 0.025, 0.035, 0.035, 0.079, 0.079, 0.072, 0.072, 0.062, 0.062, 0.107, 0.107, 0.087, 0.087, 0.089, 0.089])

def compute_oks(pred_kp, gt_kp, bbox_area):
    """관절 부위별 난이도를 반영한 정확도 점수 (OKS) 계산"""
    if pred_kp is None or len(pred_kp) == 0: return 0.0
    oks_sum, valid_count = 0, 0

    for i in range(17):
        gt_x, gt_y, vis = gt_kp[i]
        if vis > 0:
            pred_x, pred_y = pred_kp[i][0], pred_kp[i][1]
            dist_sq = (pred_x - gt_x)**2 + (pred_y - gt_y)**2
            sigma = COCO_SIGMAS[i]
            oks_sum += np.exp(-dist_sq / (2 * bbox_area * (sigma ** 2)))
            valid_count += 1
    return oks_sum / valid_count if valid_count > 0 else 0.0

def compute_detailed_pck(pred_kp, gt_kp, bbox, threshold=0.1):
    """보이는 관절과 가려진 관절을 구분하여 정확도(PCK) 계산"""
    if pred_kp is None or len(pred_kp) == 0: return 0.0, 0.0
    norm = max(bbox[2], bbox[3])

    c_vis, t_vis = 0, 0
    c_occ, t_occ = 0, 0

    for i in range(17):
        gt_x, gt_y, vis = gt_kp[i]
        if vis > 0:
            dist = np.linalg.norm([pred_kp[i][0] - gt_x, pred_kp[i][1] - gt_y])
            is_correct = dist < (threshold * norm)

            if vis == 2: # 완전히 보이는 관절
                t_vis += 1
                if is_correct: c_vis += 1
            elif vis == 1: # 옷이나 사물에 가려진 관절
                t_occ += 1
                if is_correct: c_occ += 1

    pck_vis = (c_vis / t_vis) if t_vis > 0 else 1.0
    pck_occ = (c_occ / t_occ) if t_occ > 0 else 1.0
    return pck_vis, pck_occ

def measure_fps(func, *args):
    """함수 실행 시간을 측정하여 FPS로 변환"""
    start = time.time()
    func(*args)
    end = time.time()
    elapsed = end - start
    return 1 / elapsed if elapsed > 0 else 0.0

# ==========================================
# 2. 모델 로드 및 추론 함수
# ==========================================
print("MoveNet 로드 중...")
movenet_model = hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4")
movenet = movenet_model.signatures['serving_default']

def run_movenet(image_path, img_width, img_height):
    img = cv2.imread(image_path)
    img_resized = cv2.resize(img, (192, 192))
    img_resized = tf.cast(img_resized, dtype=tf.uint8)
    outputs = movenet(tf.expand_dims(img_resized, axis=0))

    keypoints = outputs['output_0'].numpy()[0, 0]
    abs_kp = [[x * img_width, y * img_height, score] for y, x, score in keypoints]
    return np.array(abs_kp)

print("YOLOv8-Pose 로드 중...")
yolo_model = YOLO('yolov8n-pose.pt')

def run_yolo_pose(image_path):
    results = yolo_model(image_path, verbose=False)
    if len(results[0].keypoints) > 0:
        kp = results[0].keypoints.data[0].cpu().numpy()
        kp[:, 2] = 2  # 강제로 visible 처리 (또는 threshold 기반)
        return kp
    return None

# ==========================================
# 3. 종합 평가 루프 (100장)
# ==========================================
print("\n========== 속도 및 정확도 벤치마크 시작 (100장) ==========")

metrics = {
    'movenet': {'fps': [], 'oks': [], 'pck_vis': [], 'pck_occ': [], 'misses': 0},
    'yolo':    {'fps': [], 'oks': [], 'pck_vis': [], 'pck_occ': [], 'misses': 0}
}

for i, img_info in enumerate(sampled_images):
    path = os.path.join(SAVE_DIR, img_info['file_name'])
    w, h = img_info['width'], img_info['height']

    # 정답(GT) 정보 추출
    gt_anns = [ann for ann in data['annotations'] if ann['image_id'] == img_info['id'] and ann['num_keypoints'] > 0]
    if not gt_anns: continue
    best_ann = max(gt_anns, key=lambda x: x['bbox'][2] * x['bbox'][3])
    gt_kp = np.array(best_ann['keypoints']).reshape(-1, 3)
    bbox = best_ann['bbox']
    area = best_ann['area']

    # 워밍업 (첫 실행 지연 방지)
    if i == 0:
        run_movenet(path, w, h)
        run_yolo_pose(path)

    # 1. MoveNet 평가
    m_pred = run_movenet(path, w, h)
    metrics['movenet']['fps'].append(measure_fps(run_movenet, path, w, h))
    if m_pred is None:
        metrics['movenet']['misses'] += 1
    else:
        metrics['movenet']['oks'].append(compute_oks(m_pred, gt_kp, area))
        p_vis, p_occ = compute_detailed_pck(m_pred, gt_kp, bbox)
        metrics['movenet']['pck_vis'].append(p_vis)
        metrics['movenet']['pck_occ'].append(p_occ)

    # 2. YOLOv8 평가
    y_pred = run_yolo_pose(path)
    metrics['yolo']['fps'].append(measure_fps(run_yolo_pose, path))
    if y_pred is None:
        metrics['yolo']['misses'] += 1
    else:
        metrics['yolo']['oks'].append(compute_oks(y_pred, gt_kp, area))
        p_vis, p_occ = compute_detailed_pck(y_pred, gt_kp, bbox)
        metrics['yolo']['pck_vis'].append(p_vis)
        metrics['yolo']['pck_occ'].append(p_occ)

    # 진행률 출력 (20장 단위)
    if (i + 1) % 20 == 0:
        print(f" ... [{i + 1}/100] 평가 완료")

# ==========================================
# 4. 최종 결과 출력 함수
# ==========================================
def print_report(name, data_dict):
    avg_fps = np.mean(data_dict['fps'])
    miss_rate = (data_dict['misses'] / 100) * 100
    avg_oks = np.mean(data_dict['oks']) * 100
    pck_v = np.mean(data_dict['pck_vis']) * 100
    pck_o = np.mean(data_dict['pck_occ']) * 100

    print(f"\n[{name}]")
    print(f" ⚡ 속도 관점")
    print(f"    - 평균 처리 속도 : {avg_fps:.1f} FPS")
    print(f"    - 객체 탐지 실패율: {miss_rate:.1f} %")
    print(f" 🎯 정확도 관점 (높을수록 좋음)")
    print(f"    - OKS 종합 점수  : {avg_oks:.1f} 점")
    print(f"    - PCK (보이는 관절): {pck_v:.1f} %")
    print(f"    - PCK (가려진 관절): {pck_o:.1f} %")

print("\n" + "="*50)
print(" 🏆 속도 및 정확도 벤치마크 최종 리포트 🏆")
print("="*50)
print_report("MoveNet (Lightning)", metrics['movenet'])
print("-" * 50)
print_report("YOLOv8-Pose (Nano)", metrics['yolo'])
print("="*50)

In [ ]:
# ==========================================
# 0. 라이브러리 설치 (최초 1회 실행 시 시간 소요)
# ==========================================
!pip install -q tensorflow tensorflow-hub opencv-python ultralytics
!pip install --upgrade setuptools packaging
!wget -nc -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -n -q annotations_trainval2017.zip

# RTMPose 설치 (원치 않으시면 아래 3줄은 주석 처리 후 실행해도 됩니다)
#!pip install -U -q openmim
#!mim install -q mmengine "mmcv>=2.0.0" "mmdet>=3.0.0"
#!pip install -q mmpose

import json
import os
import random
import time
import numpy as np
import cv2
import tensorflow as tf
import tensorflow_hub as hub
from ultralytics import YOLO
import requests

# RTMPose 임포트 시도
try:
    from mmpose.apis import MMPoseInferencer
    HAS_RTMPOSE = True
except ImportError:
    HAS_RTMPOSE = False
    print("⚠️ MMPose가 설치되지 않아 RTMPose 평가는 건너뜁니다.")

# ==========================================
# 1. 데이터셋 준비 (100장)
# ==========================================
random.seed(42)
ANNOT_PATH = "annotations/person_keypoints_val2017.json"
SAVE_DIR = "images"
os.makedirs(SAVE_DIR, exist_ok=True)

with open(ANNOT_PATH, 'r') as f:
    data = json.load(f)

valid_image_ids = set([ann['image_id'] for ann in data['annotations'] if ann['num_keypoints'] > 0])
filtered_images = [img for img in data['images'] if img['id'] in valid_image_ids]
sampled_images = random.sample(filtered_images, 100)

print("테스트 이미지 100장 다운로드 중...")
for img_info in sampled_images:
    file_name = img_info['file_name']
    img_path = os.path.join(SAVE_DIR, file_name)
    if not os.path.exists(img_path):
        with open(img_path, 'wb') as f:
            f.write(requests.get(f"http://images.cocodataset.org/val2017/{file_name}").content)
print("다운로드 완료!\n")

# ==========================================
# 2. 평가 지표 함수: PCK, Mean Error, FPS
# ==========================================
def compute_metrics(pred_kp, gt_kp, bbox, threshold=0.1):
    """PCK(정확도) 및 Mean Error(픽셀 오차) 동시 계산"""
    if pred_kp is None or len(pred_kp) == 0: return 0.0, 0.0

    norm = max(bbox[2], bbox[3]) # 바운딩 박스 최대 길이
    c_correct, total_valid = 0, 0
    distances = []

    for i in range(17):
        gt_x, gt_y, vis = gt_kp[i]
        if vis > 0: # 화면에 존재하는 관절만 평가
            total_valid += 1
            pred_x, pred_y = pred_kp[i][0], pred_kp[i][1]
            dist = np.linalg.norm([pred_x - gt_x, pred_y - gt_y]) # 유클리드 거리 (픽셀 단위)
            distances.append(dist)

            # PCK 검사
            if dist < (threshold * norm):
                c_correct += 1

    pck = (c_correct / total_valid) if total_valid > 0 else 0.0
    mean_error = np.mean(distances) if len(distances) > 0 else 0.0
    return pck, mean_error

# ==========================================
# 3. 모델 로드 및 추론 함수 (⭐ 독거노인 필터 적용)
# ==========================================

# --- 1) MoveNet MultiPose ---
print("MoveNet MultiPose 로드 중...")
movenet_multi = hub.load("https://tfhub.dev/google/movenet/multipose/lightning/1").signatures['serving_default']

def run_movenet_multi(image_path, img_w, img_h):
    img = cv2.imread(image_path)
    # MultiPose는 256x256 등 32의 배수 입력 권장
    img_resized = cv2.resize(img, (256, 256)).astype(np.int32)
    start = time.time()
    outputs = movenet_multi(tf.expand_dims(img_resized, axis=0))
    latency = time.time() - start

    # [1, 6, 56] 반환 (최대 6명. 각 인물당 51개 좌표값 + 5개 바운딩박스값)
    people = outputs['output_0'].numpy()[0]

    best_kp = None
    max_area = 0
    for person in people:
        score = person[55] # 인물 신뢰도
        if score > 0.1:
            ymin, xmin, ymax, xmax = person[51:55]

            # 🔥 정규화 → 픽셀 좌표 변환
            ymin, ymax = ymin * img_h, ymax * img_h
            xmin, xmax = xmin * img_w, xmax * img_w

            area = (ymax - ymin) * (xmax - xmin)
            # ⭐ 가장 크게 잡힌 1명만 필터링
            if area > max_area:
                max_area = area
                kpts = person[:51].reshape(17, 3) # [y, x, conf] 정규화값
                best_kp = [[kp[1]*img_w, kp[0]*img_h, kp[2]] for kp in kpts]
    return np.array(best_kp) if best_kp else None, latency

# --- 2) YOLOv8n-Pose ---
print("YOLOv8n-Pose 로드 중...")
yolo_model = YOLO('yolov8n-pose.pt')

def run_yolo(image_path):
    start = time.time()
    results = yolo_model(image_path, verbose=False)
    latency = time.time() - start

    # 수정된 코드
    if len(results[0].boxes) > 0 and results[0].keypoints is not None:
        boxes = results[0].boxes.data.cpu().numpy()
        keypoints = results[0].keypoints.data.cpu().numpy()

        # ⭐ 가장 크게 잡힌 1명만 필터링
        areas = [(box[2]-box[0]) * (box[3]-box[1]) for box in boxes]
        best_idx = np.argmax(areas)
        kp = keypoints[best_idx]
        if kp.shape[-1] != 3:
          kp = kp.reshape(17, 3)

        return kp, latency

    return None, latency

# --- 3) RTMPose-t ---
if HAS_RTMPOSE:
    print("RTMPose-t 로드 중...")
    rtm_inferencer = MMPoseInferencer(
        pose2d='rtmpose-t_8xb256-420e_coco-256x192',
        show_progress=False
    )

def run_rtmpose(image_path):
    if not HAS_RTMPOSE:
        return None, 0.0

    start = time.time()
    result = next(rtm_inferencer(image_path))
    latency = time.time() - start

    instances = result['predictions'][0]

    if len(instances) > 0:
        # ⭐ 가장 크게 잡힌 1명만 필터링
        # 수정된 코드
        best_inst = max(instances, key=lambda x: (x['bbox'][2] - x['bbox'][0]) * (x['bbox'][3] - x['bbox'][1]))
        # x, y, score 병합
        kpts = [[kp[0], kp[1], score] for kp, score in zip(best_inst['keypoints'], best_inst['keypoint_scores'])]
        return np.array(kpts), latency
    return None, latency

# ==========================================
# 4. 100장 벤치마크 평가 루프
# ==========================================
print("\n========== [독거노인 환경 최적화] 모델 벤치마크 시작 ==========")

metrics = {
    'movenet': {'fps': [], 'pck': [], 'error': [], 'misses': 0},
    'yolo':    {'fps': [], 'pck': [], 'error': [], 'misses': 0},
    'rtmpose': {'fps': [], 'pck': [], 'error': [], 'misses': 0}
}

for i, img_info in enumerate(sampled_images):
    path = os.path.join(SAVE_DIR, img_info['file_name'])
    w, h = img_info['width'], img_info['height']

    gt_anns = [ann for ann in data['annotations'] if ann['image_id'] == img_info['id'] and ann['num_keypoints'] > 0]
    if not gt_anns: continue

    # 정답(GT) 중 화면에서 가장 크게 잡힌 '메인 타겟(독거노인 모사)' 추출
    best_ann = max(gt_anns, key=lambda x: x['bbox'][2] * x['bbox'][3])
    gt_kp = np.array(best_ann['keypoints']).reshape(-1, 3)
    gt_bbox = best_ann['bbox']

    # 워밍업
    if i == 0:
        run_movenet_multi(path, w, h); run_yolo(path)
        if HAS_RTMPOSE: run_rtmpose(path)

    # 1. MoveNet
    pred, lat = run_movenet_multi(path, w, h)
    metrics['movenet']['fps'].append(1/(lat + 1e-6))
    if pred is None: metrics['movenet']['misses'] += 1
    else:
        pck, err = compute_metrics(pred, gt_kp, gt_bbox)
        metrics['movenet']['pck'].append(pck); metrics['movenet']['error'].append(err)

    # 2. YOLOv8
    pred, lat = run_yolo(path)
    metrics['yolo']['fps'].append(1/(lat + 1e-6))
    if pred is None: metrics['yolo']['misses'] += 1
    else:
        pck, err = compute_metrics(pred, gt_kp, gt_bbox)
        metrics['yolo']['pck'].append(pck); metrics['yolo']['error'].append(err)

    # 3. RTMPose
    if HAS_RTMPOSE:
        pred, lat = run_rtmpose(path)
        metrics['rtmpose']['fps'].append(1/(lat + 1e-6))
        if pred is None: metrics['rtmpose']['misses'] += 1
    else:
        pck, err = compute_metrics(pred, gt_kp, gt_bbox)
        metrics['rtmpose']['pck'].append(pck); metrics['rtmpose']['error'].append(err)

    if (i + 1) % 20 == 0: print(f" ... [{i + 1}/100] 평가 완료")

# ==========================================
# 5. 최종 결과 리포트
# ==========================================
def print_report(name, data_dict):
    avg_fps = np.mean(data_dict['fps']) if data_dict['fps'] else 0
    pck = np.mean(data_dict['pck']) * 100 if data_dict['pck'] else 0
    avg_err = np.mean(data_dict['error']) if data_dict['error'] else 0

    print(f"\n[{name}]")
    print(f" ⚡ 속도   (FPS)       : {avg_fps:.1f} 프레임/초 (높을수록 실시간성에 유리)")
    print(f" 🎯 정확도 (PCK)       : {pck:.1f} % (정답 반경 내에 들어온 관절의 비율, 높을수록 좋음)")
    print(f" 📏 오차   (Mean Error): {avg_err:.1f} 픽셀 (실제 위치와 어긋난 평균 거리, 낮을수록 정밀함)")

print("\n" + "="*60)
print(" 🏆 모델별 성능 벤치마크 결과 (독거노인 1인 필터 적용) 🏆")
print("="*60)
print_report("MoveNet MultiPose (Lightning)", metrics['movenet'])
print("-" * 60)
print_report("YOLOv8-Pose (Nano)", metrics['yolo'])
print("-" * 60)
if HAS_RTMPOSE:
    print_report("RTMPose-t (OpenMMLab)", metrics['rtmpose'])
else:
    print("\n[RTMPose-t (OpenMMLab)]\n - 설치 생략으로 인해 평가되지 않음.")
print("="*60)

AttributeError: module 'pkgutil' has no attribute 'ImpImporter'